# PyEcharts 数据可视化学习

PyEcharts 是 ECharts 的 Python 封装库，用于生成交互式图表。与 Matplotlib 的静态图不同，PyEcharts 支持工具栏缩放、数据区域缩放、图表切换等丰富的交互功能。本教程涵盖基础图表、组合布局和主题配置，重点讲解如何使用 Grid 组件实现多图表的合理布局，确保图表之间不出现重叠。

In [69]:
# 导入必要的库
from pyecharts.charts import Line, Bar, Pie, Scatter, HeatMap, Grid, Page, Tab, Timeline, Radar, WordCloud, EffectScatter, Liquid, Gauge, Sunburst
from pyecharts import options as opts
from pyecharts.globals import ThemeType, SymbolType
import pandas as pd
import numpy as np

# 加载示例数据
weather = pd.read_csv("data/weather.csv")
penguins = pd.read_csv("data/penguins.csv")
penguins.dropna(inplace=True)
house = pd.read_csv("data/house_sales.csv")

print(f"天气数据: {weather.shape}")
print(f"企鹅数据: {penguins.shape}")
print(f"房产数据: {house.shape}")

天气数据: (1461, 6)
企鹅数据: (333, 7)
房产数据: (106118, 12)


## 8.1 基础图表

PyEcharts 提供了丰富的基础图表类型，包括折线图、柱状图、饼图、散点图等。每种图表都支持丰富的配置选项，如标题、图例、坐标轴、标注等。

### 折线图（Line）

折线图用于展示数据随时间或其他连续变量变化的趋势，非常适合时间序列数据的可视化。

In [70]:
# 折线图：2015年月平均气温趋势
weather['date'] = pd.to_datetime(weather['date'])
df_2015 = weather[weather['date'].dt.year == 2015].copy()
df_2015['month'] = df_2015['date'].dt.month
monthly_temp = df_2015.groupby('month')[['temp_max', 'temp_min']].mean()

line = (
    Line(init_opts=opts.InitOpts(width="800px", height="400px"))
    .add_xaxis(monthly_temp.index.tolist())
    .add_yaxis("最高气温", monthly_temp['temp_max'].round(1).tolist(), 
               markpoint_opts=opts.MarkPointOpts(data=[opts.MarkPointItem(type_="max")]))
    .add_yaxis("最低气温", monthly_temp['temp_min'].round(1).tolist(), 
               markpoint_opts=opts.MarkPointOpts(data=[opts.MarkPointItem(type_="min")]))
    .set_global_opts(
        title_opts=opts.TitleOpts(title="2015年月平均气温趋势", subtitle="单位：摄氏度"),
        xaxis_opts=opts.AxisOpts(name="月份"),
        yaxis_opts=opts.AxisOpts(name="气温 (°C)"),
        tooltip_opts=opts.TooltipOpts(trigger="axis")
    )
)
line.render_notebook()

### 柱状图（Bar）

柱状图适合展示分类数据之间的对比关系，可以通过设置不同的柱子颜色、宽度等来增强可读性。

In [71]:
# 柱状图：各岛屿企鹅数量
island_counts = penguins['island'].value_counts()

bar = (
    Bar(init_opts=opts.InitOpts(width="700px", height="400px"))
    .add_xaxis(island_counts.index.tolist())
    .add_yaxis("数量", island_counts.values.tolist())
    .set_global_opts(
        title_opts=opts.TitleOpts(title="各岛屿企鹅数量"),
        xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=0)),
        yaxis_opts=opts.AxisOpts(name="数量"),
        tooltip_opts=opts.TooltipOpts(trigger="axis")
    )
    .set_series_opts(
        label_opts=opts.LabelOpts(position="top"),
        itemstyle_opts=opts.ItemStyleOpts(color="#4ECDC4")
    )
)
bar.render_notebook()

### 饼图（Pie）

饼图适合展示各分类占总体的比例关系，通过 radius 参数可以创建环形饼图。

In [72]:
# 饼图：天气类型分布
weather_counts = weather['weather'].value_counts()

pie = (
    Pie(init_opts=opts.InitOpts(width="600px", height="400px"))
    .add(
        "",
        [list(z) for z in zip(weather_counts.index.tolist(), [int(x) for x in weather_counts.values.tolist()])],
        radius=["30%", "70%"],
        label_opts=opts.LabelOpts(formatter="{b}: {d}%")
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="天气类型分布", pos_left="center"),
        legend_opts=opts.LegendOpts(orient="vertical", pos_left="left", pos_top="middle")
    )
)
pie.render_notebook()


### 散点图（Scatter）

散点图用于展示两个连续变量之间的关系，通过颜色区分不同类别可以同时展示多组数据。

In [73]:
# 散点图：企鹅喙长与喙深关系
from pyecharts import options as opts
from pyecharts.globals import JsCode

# 使用 overlap 方式：每个物种一个独立的 Scatter，然后叠加
scatter_adelie = (
    Scatter()
    .add_xaxis([float(x) for x in penguins[penguins['species']=='Adelie']['bill_length_mm'].tolist()])
    .add_yaxis('Adelie', [float(y) for y in penguins[penguins['species']=='Adelie']['bill_depth_mm'].tolist()])
    .set_series_opts(label_opts=opts.LabelOpts(is_show=False))
)

scatter_chinstrap = (
    Scatter()
    .add_xaxis([float(x) for x in penguins[penguins['species']=='Chinstrap']['bill_length_mm'].tolist()])
    .add_yaxis('Chinstrap', [float(y) for y in penguins[penguins['species']=='Chinstrap']['bill_depth_mm'].tolist()])
    .set_series_opts(label_opts=opts.LabelOpts(is_show=False))
)

scatter_gentoo = (
    Scatter()
    .add_xaxis([float(x) for x in penguins[penguins['species']=='Gentoo']['bill_length_mm'].tolist()])
    .add_yaxis('Gentoo', [float(y) for y in penguins[penguins['species']=='Gentoo']['bill_depth_mm'].tolist()])
    .set_series_opts(label_opts=opts.LabelOpts(is_show=False))
)

# 叠加所有系列
scatter_chart = scatter_adelie
for s in [scatter_chinstrap, scatter_gentoo]:
    scatter_chart = scatter_chart.overlap(s)

scatter_chart.set_global_opts(
    title_opts=opts.TitleOpts(title="企鹅喙长与喙深关系"),
    xaxis_opts=opts.AxisOpts(name="喙长 (mm)"),
    yaxis_opts=opts.AxisOpts(name="喙深 (mm)"),
    legend_opts=opts.LegendOpts(pos_left="right"),
    tooltip_opts=opts.TooltipOpts(
        trigger="item",
        formatter=JsCode("function(params){return params.seriesName + '<br/>喙长: ' + params.data[0] + ' mm<br/>喙深: ' + params.data[1] + ' mm';}")
    )
)
scatter_chart.render_notebook()


## 8.2 使用真实数据的图表

下面我们使用真实数据集展示更复杂的数据可视化场景。

### 城市房价对比

使用 house_sales.csv 数据分析各省份的平均房价。

In [74]:
# 处理房价数据：各城市平均单价
house_clean = house.dropna(subset=['unit'])
house_clean = house_clean[house_clean['unit'].str.contains('元')]
house_clean['price_num'] = house_clean['unit'].str.extract(r'(\d+)').astype(float)
city_price = house_clean.groupby('province')['price_num'].mean().sort_values(ascending=True).tail(10)

bar2 = (
    Bar(init_opts=opts.InitOpts(width="850px", height="450px"))
    .add_xaxis(city_price.index.tolist())
    .add_yaxis("平均单价", [round(x, 0) for x in city_price.values.tolist()], category_gap="50%")
    .set_global_opts(
        title_opts=opts.TitleOpts(title="各省份房产均价对比 (TOP 10)"),
        xaxis_opts=opts.AxisOpts(name="省份", axislabel_opts=opts.LabelOpts(rotate=30)),
        yaxis_opts=opts.AxisOpts(name="单价 (元/㎡)"),
        tooltip_opts=opts.TooltipOpts(trigger="axis")
    )
    .set_series_opts(
        label_opts=opts.LabelOpts(position="top"),
        itemstyle_opts=opts.ItemStyleOpts(color="#FF6B6B")
    )
)
bar2.render_notebook()

### 月度气温趋势

使用 weather.csv 展示气温随时间的变化趋势。

In [75]:
# 气温走势折线图
df_sample = weather[weather['date'].dt.year == 2015].head(30)
line2 = (
    Line(init_opts=opts.InitOpts(width="900px", height="400px"))
    .add_xaxis([d.strftime('%m-%d') for d in df_sample['date']])
    .add_yaxis(
        "最高气温",
        df_sample['temp_max'].tolist(),
        markline_opts=opts.MarkLineOpts(data=[opts.MarkLineItem(type_="average")])
    )
    .add_yaxis("最低气温", df_sample['temp_min'].tolist())
    .set_global_opts(
        title_opts=opts.TitleOpts(title="2015年1月气温走势"),
        xaxis_opts=opts.AxisOpts(name="日期", axislabel_opts=opts.LabelOpts(rotate=45)),
        yaxis_opts=opts.AxisOpts(name="气温 (°C)"),
        tooltip_opts=opts.TooltipOpts(trigger="axis")
    )
)
line2.render_notebook()

### 热力图

热力图适合展示二维数据矩阵中数值的大小分布，常用于相关性分析。

In [76]:
# 热力图：企鹅身体指标相关性
num_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
corr = penguins[num_cols].corr()

hm = (
    HeatMap(init_opts=opts.InitOpts(width="700px", height="500px"))
    .add_xaxis(corr.columns.tolist())
    .add_yaxis(
        "",
        corr.index.tolist(),
        [[i, j, round(corr.iloc[i, j], 2)] for i in range(len(corr)) for j in range(len(corr))]
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="企鹅身体指标相关性热力图"),
        visualmap_opts=opts.VisualMapOpts(min_=-1, max_=1, is_calculable=True),
        tooltip_opts=opts.TooltipOpts(trigger="item")
    )
)
hm.render_notebook()

### 堆叠柱状图

堆叠柱状图可以展示各分类的组成部分及其总量。

In [77]:
# 堆叠柱状图：各城市房屋面积分布
# 简化处理：选择主要户型，按城市展示平均面积
house_clean['area_num'] = house_clean['area'].str.replace('㎡', '', regex=False).astype(float)

# 选择数据最多的前5个城市
top_cities = house_clean['city'].value_counts().head(5).index.tolist()
# 选择主要户型（数量最多的前6种）
top_rooms = house_clean['rooms'].value_counts().head(6).index.tolist()

# 筛选数据
filtered = house_clean[house_clean['city'].isin(top_cities) & house_clean['rooms'].isin(top_rooms)]
city_areas = filtered.groupby(['city', 'rooms'])['area_num'].mean().unstack()
city_areas = city_areas.fillna(0)

bar3 = (
    Bar(init_opts=opts.InitOpts(width="900px", height="500px"))
    .add_xaxis(city_areas.index.tolist())
)
for room in top_rooms:
    if room in city_areas.columns:
        bar3.add_yaxis(f"{room}", [round(x, 1) for x in city_areas[room].tolist()], stack=True)

bar3.set_series_opts(label_opts=opts.LabelOpts(is_show=False))
bar3.set_global_opts(
    title_opts=opts.TitleOpts(title="主要城市房屋面积分布"),
    xaxis_opts=opts.AxisOpts(name="城市", axislabel_opts=opts.LabelOpts(rotate=20)),
    yaxis_opts=opts.AxisOpts(name="面积 (㎡)"),
    legend_opts=opts.LegendOpts(pos_bottom="bottom"),
    tooltip_opts=opts.TooltipOpts(trigger="axis")
)
bar3.render_notebook()


## 8.3 Grid 组合布局

**重要**: Grid 组件是实现多图表合理布局的核心工具。通过设置 pos_top、pos_bottom、pos_left、pos_right 等参数（使用百分比），可以精确控制每个图表在容器中的位置，避免元素重叠。

**防重叠原则**：
1. 上方图表的 pos_bottom 值要小于下方图表的 pos_top 值
2. 相邻图表之间保持适当间距（建议 2-5%）
3. 使用 grid_gap 参数增加格子之间的间距

### Grid 上下布局

上下布局时，上方图表使用较大的 pos_bottom 值，下方图表使用较大的 pos_top 值，确保不重叠。

In [78]:
# Grid 上下布局：上方折线图 + 下方柱状图
# 注意：上方图表 pos_bottom="52%"，下方图表 pos_top="55%"，中间留有 3% 的间距

line_g = (
    Line(init_opts=opts.InitOpts(width="850px", height="300px"))
    .add_xaxis(["1月", "2月", "3月", "4月", "5月", "6月"])
    .add_yaxis("蒸发量", [50, 70, 90, 120, 150, 180])
    .add_yaxis("降水量", [60, 80, 100, 130, 160, 200])
    .set_global_opts(
        title_opts=opts.TitleOpts(title="气象数据对比", subtitle="2015年月度统计", pos_left="center", pos_top="0%"),
        tooltip_opts=opts.TooltipOpts(trigger="axis")
    )
)

bar_g = (
    Bar(init_opts=opts.InitOpts(width="850px", height="280px"))
    .add_xaxis(["1月", "2月", "3月", "4月", "5月", "6月"])
    .add_yaxis("蒸发量", [50, 70, 90, 120, 150, 180])
    .add_yaxis("降水量", [60, 80, 100, 130, 160, 200])
    .set_series_opts(
        label_opts=opts.LabelOpts(position="top"),
        itemstyle_opts=opts.ItemStyleOpts(color="#45B7D1")
    )
)

grid = (
    Grid(init_opts=opts.InitOpts(width="900px", height="800px"))
    .add(line_g, grid_opts=opts.GridOpts(pos_top="10%", pos_bottom="52%", pos_left="5%", pos_right="5%"))
    .add(bar_g, grid_opts=opts.GridOpts(pos_top="55%", pos_bottom="10%", pos_left="5%", pos_right="5%"))
)
grid.render_notebook()


### Grid 左右布局

左右布局时，左侧图表使用较大的 pos_right 值，右侧图表使用较大的 pos_left 值。

In [82]:
# 左侧散点图
scatter_left = (
    Scatter()
    .add_xaxis([10, 20, 30, 40, 50])
    .add_yaxis("A组", [20, 30, 40, 50, 60], xaxis_index=0, yaxis_index=0)
    .add_yaxis("B组", [30, 40, 50, 60, 70], xaxis_index=0, yaxis_index=0)
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show=False),
        itemstyle_opts=opts.ItemStyleOpts(opacity=0.8)
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="分组对比", pos_left="10%"),
        tooltip_opts=opts.TooltipOpts(trigger="item"),
        legend_opts=opts.LegendOpts(pos_top="5%"),
        xaxis_opts=opts.AxisOpts(
            type_="value",
            name="指标"
        ),
        yaxis_opts=opts.AxisOpts(
            type_="value",
            name="数值"
        )
    )
)
# 右侧柱状图
bar_right = (
    Bar()
    .add_xaxis(["A组", "B组", "C组"])
    .add_yaxis("数值", [45, 55, 48], xaxis_index=1, yaxis_index=1)
    .set_series_opts(
        label_opts=opts.LabelOpts(position="top"),
        itemstyle_opts=opts.ItemStyleOpts(color="#96CEB4")
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="分组统计", pos_left="62%"),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
        legend_opts=opts.LegendOpts(pos_bottom="2%"),
        xaxis_opts=opts.AxisOpts(type_="category"),
        yaxis_opts=opts.AxisOpts(name="数值")
    )
)
# Grid 布局
grid_lr = (
    Grid(init_opts=opts.InitOpts(width="980px", height="500px"))
    .add(
        scatter_left,
        grid_opts=opts.GridOpts(
            pos_left="8%",
            pos_right="55%",
            pos_top="18%",
            pos_bottom="14%"
        ),
        is_control_axis_index=True
    )
    .add(
        bar_right,
        grid_opts=opts.GridOpts(
            pos_left="60%",
            pos_right="8%",
            pos_top="18%",
            pos_bottom="14%"
        ),
        is_control_axis_index=True
    )
)
grid_lr.render_notebook()

### Grid 四宫格仪表盘

四宫格布局将页面分成 2x2 的网格，每个格子放置一个图表。

In [87]:
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Scatter, Pie, Grid

# 左上：柱状图
bar1 = (
    Bar()
    .add_xaxis(["衬衫", "毛衣", "裙子", "裤子", "风衣"])
    .add_yaxis("数量", [20, 30, 40, 35, 25], xaxis_index=0, yaxis_index=0)
    .add_yaxis("库存", [50, 40, 30, 45, 55], xaxis_index=0, yaxis_index=0)
    .set_series_opts(label_opts=opts.LabelOpts(is_show=False))
    .set_global_opts(
        title_opts=opts.TitleOpts(title="服装销量", pos_left="8%", pos_top="3%"),
        legend_opts=opts.LegendOpts(pos_left="20%", pos_top="3%"),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
        toolbox_opts=opts.ToolboxOpts(is_show=False),
        xaxis_opts=opts.AxisOpts(type_="category"),
        yaxis_opts=opts.AxisOpts(type_="value")
    )
)

# 右上：折线图
line1 = (
    Line()
    .add_xaxis(["周一", "周二", "周三", "周四", "周五"])
    .add_yaxis("销量", [120, 200, 150, 80, 70], xaxis_index=1, yaxis_index=1)
    .add_yaxis("访问量", [300, 350, 280, 400, 380], xaxis_index=1, yaxis_index=1)
    .set_series_opts(label_opts=opts.LabelOpts(is_show=False))
    .set_global_opts(
        title_opts=opts.TitleOpts(title="访问趋势", pos_left="58%", pos_top="3%"),
        legend_opts=opts.LegendOpts(pos_left="70%", pos_top="3%"),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
        toolbox_opts=opts.ToolboxOpts(is_show=False),
        xaxis_opts=opts.AxisOpts(type_="category"),
        yaxis_opts=opts.AxisOpts(type_="value")
    )
)

# 左下：散点图
scatter1 = (
    Scatter()
    .add_xaxis(["A", "B", "C", "D", "E"])
    .add_yaxis("数据", [50, 60, 70, 80, 90], xaxis_index=2, yaxis_index=2)
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show=False),
        symbol_size=14
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="数据分布", pos_left="8%", pos_top="53%"),
        legend_opts=opts.LegendOpts(pos_left="20%", pos_top="53%"),
        tooltip_opts=opts.TooltipOpts(trigger="item"),
        toolbox_opts=opts.ToolboxOpts(is_show=False),
        xaxis_opts=opts.AxisOpts(type_="category"),
        yaxis_opts=opts.AxisOpts(type_="value")
    )
)

# 右下：饼图
pie1 = (
    Pie()
    .add(
        "",
        [["北京", 45], ["上海", 35], ["广州", 20]],
        center=["75%", "74%"],
        radius=["12%", "22%"]
    )
    .set_series_opts(
        label_opts=opts.LabelOpts(formatter="{b}: {c}")
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="城市占比", pos_left="58%", pos_top="53%"),
        legend_opts=opts.LegendOpts(
            pos_left="86%",
            pos_top="66%",
            orient="vertical"
        ),
        tooltip_opts=opts.TooltipOpts(trigger="item"),
        toolbox_opts=opts.ToolboxOpts(is_show=False)
    )
)

grid4 = (
    Grid(init_opts=opts.InitOpts(width="1000px", height="800px"))
    .add(
        bar1,
        grid_opts=opts.GridOpts(
            pos_left="8%", pos_right="56%",
            pos_top="12%", pos_bottom="56%"
        ),
        is_control_axis_index=True
    )
    .add(
        line1,
        grid_opts=opts.GridOpts(
            pos_left="58%", pos_right="8%",
            pos_top="12%", pos_bottom="56%"
        ),
        is_control_axis_index=True
    )
    .add(
        scatter1,
        grid_opts=opts.GridOpts(
            pos_left="8%", pos_right="56%",
            pos_top="62%", pos_bottom="10%"
        ),
        is_control_axis_index=True
    )
    .add(
        pie1,
        grid_opts=opts.GridOpts(
            pos_left="58%", pos_right="8%",
            pos_top="62%", pos_bottom="10%"
        )
    )
)

grid4.render_notebook()

# 叠加图表：折线 + 柱状混合

In [88]:
# 叠加图表：折线 + 柱状混合
from pyecharts.charts import Line, Bar

overlap_base = (
    Bar(init_opts=opts.InitOpts(width="800px", height="450px"))
    .add_xaxis(["周一", "周二", "周三", "周四", "周五", "周六", "周日"])
    .add_yaxis("咖啡销量", [120, 150, 90, 180, 200, 250, 280])
    .add_yaxis("茶销量", [80, 60, 100, 70, 90, 120, 110])
    .set_series_opts(
        label_opts=opts.LabelOpts(position="top"),
        itemstyle_opts=opts.ItemStyleOpts(color="#FFEAA7")
    )
)

line_overlay = (
    Line()
    .add_xaxis(["周一", "周二", "周三", "周四", "周五", "周六", "周日"])
    .add_yaxis("总利润", [200, 210, 190, 250, 290, 370, 390])
    .set_series_opts(
        linestyle_opts=opts.LineStyleOpts(width=3, color="#FF6B6B"),
        label_opts=opts.LabelOpts(is_show=True, position="top")
    )
)

overlap_chart = overlap_base.overlap(line_overlay)
overlap_chart.set_global_opts(
    title_opts=opts.TitleOpts(title="饮品销量与利润对比"),
    tooltip_opts=opts.TooltipOpts(trigger="axis")
)
overlap_chart.render_notebook()

## 8.4 组合组件

PyEcharts 提供了 Page、Tab、Timeline 等组合组件，用于组织和展示多个图表。

### Page - 多图表分页

Page 组件允许将多个图表放在同一个页面中，支持滚动浏览。

In [89]:
# Page：多图表分页展示
page = Page(layout=Page.SimplePageLayout)

line_p = (
    Line()
    .add_xaxis(["1月", "2月", "3月", "4月"])
    .add_yaxis("产品A", [100, 120, 130, 150])
    .add_yaxis("产品B", [80, 90, 100, 110])
    .set_global_opts(title_opts=opts.TitleOpts(title="产品销售趋势"))
)

bar_p = (
    Bar()
    .add_xaxis(["手机", "电脑", "平板", "耳机"])
    .add_yaxis("销量", [300, 450, 200, 150])
    .add_yaxis("库存", [100, 200, 300, 250])
    .set_global_opts(title_opts=opts.TitleOpts(title="电子产品销售统计"))
)

pie_p = (
    Pie()
    .add("", [["已完成", 60], ["进行中", 30], ["未开始", 10]], radius=["40%", "70%"])
    .set_global_opts(title_opts=opts.TitleOpts(title="项目进度"))
)

page.add(line_p, bar_p, pie_p)
page.render_notebook()

### Tab - 标签页切换

Tab 组件通过标签页的形式展示多个图表，节省页面空间。

In [15]:
# Tab：标签页切换
tab = Tab()

line_tab = (
    Line()
    .add_xaxis(["周一", "周二", "周三", "周四", "周五"])
    .add_yaxis("本周", [120, 140, 150, 170, 190])
    .add_yaxis("上周", [100, 110, 120, 130, 140])
    .set_global_opts(
        title_opts=opts.TitleOpts(title="销售趋势对比"),
        tooltip_opts=opts.TooltipOpts(trigger="axis")
    )
)

bar_tab = (
    Bar()
    .add_xaxis(["北京", "上海", "广州", "深圳"])
    .add_yaxis("销售额", [5000, 6000, 4500, 5500])
    .add_yaxis("利润", [1500, 1800, 1200, 1600])
    .set_global_opts(
        title_opts=opts.TitleOpts(title="城市销售对比"),
        tooltip_opts=opts.TooltipOpts(trigger="axis")
    )
)

pie_tab = (
    Pie()
    .add("", [["已完成", 60], ["进行中", 30], ["未开始", 10]], radius=["40%", "70%"])
    .set_global_opts(title_opts=opts.TitleOpts(title="项目进度"))
)

tab.add(line_tab, "趋势图")
tab.add(bar_tab, "对比图")
tab.add(pie_tab, "进度图")
tab.render_notebook()

### Timeline - 时间线动画

Timeline 组件允许按时间顺序展示数据的变化，支持自动播放动画。

In [16]:
# Timeline：时间线动画
tl = Timeline(init_opts=opts.InitOpts(width="800px", height="400px"))
tl.add_schema(
    play_interval=2000,  # 播放间隔 2 秒
    is_auto_play=True,   # 自动播放
    is_loop_play=True    # 循环播放
)

for year in [2013, 2014, 2015]:
    line_t = (
        Line()
        .add_xaxis(["1月", "2月", "3月", "4月"])
        .add_yaxis("销量", [100 + year * 10, 120 + year * 10, 130 + year * 10, 150 + year * 10])
        .add_yaxis("利润", [80 + year * 8, 90 + year * 8, 100 + year * 8, 120 + year * 8])
        .set_global_opts(title_opts=opts.TitleOpts(title=f"{year}年销售数据", pos_left="center"))
    )
    tl.add(line_t, f"{year}年")

tl.render_notebook()

## 8.5 高级图表

PyEcharts 还提供了许多高级图表类型，包括雷达图、词云、水球图、仪表盘等。

### 雷达图（Radar）

雷达图适合展示多维度数据的综合对比。

In [17]:
# 雷达图：城市发展对比
radar = (
    Radar(init_opts=opts.InitOpts(width="700px", height="500px"))
    .add_schema(
        schema=[
            opts.RadarIndicatorItem(name="经济", max_=100),
            opts.RadarIndicatorItem(name="交通", max_=100),
            opts.RadarIndicatorItem(name="环境", max_=100),
            opts.RadarIndicatorItem(name="教育", max_=100),
            opts.RadarIndicatorItem(name="医疗", max_=100)
        ],
        splitline_opt=opts.SplitLineOpts(is_show=True),
        splitarea_opt=opts.SplitAreaOpts(is_show=True)
    )
    .add(
        "北京",
        [[85, 90, 80, 75, 88]],
        color="#1f77b4",
        areastyle_opts=opts.AreaStyleOpts(opacity=0.3)
    )
    .add(
        "上海",
        [[90, 85, 85, 80, 86]],
        color="#ff7f0e",
        areastyle_opts=opts.AreaStyleOpts(opacity=0.3)
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="城市发展雷达图"),
        legend_opts=opts.LegendOpts(pos_left="right")
    )
)
radar.render_notebook()

### 词云（WordCloud）

词云适合展示文本数据中高频关键词的可视化。

In [18]:
# 词云：技术关键词
words = [
    ("Python", 10000), ("数据分析", 8000), ("机器学习", 7500),
    ("可视化", 6000), ("Pandas", 5500), ("PyEcharts", 5000),
    ("NumPy", 4500), ("Scikit-learn", 4000), ("深度学习", 3500),
    ("TensorFlow", 3000), ("PyTorch", 2800), ("NLP", 2600),
    ("大数据", 2400), ("云计算", 2200), ("区块链", 2000)
]

wordcloud = (
    WordCloud(init_opts=opts.InitOpts(width="800px", height="500px"))
    .add(
        "",
        words,
        word_size_range=[20, 80],
        shape="diamond",
        textstyle_opts=opts.TextStyleOpts(font_family="Microsoft YaHei")
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="技术关键词词云"),
        tooltip_opts=opts.TooltipOpts(trigger="item")
    )
)
wordcloud.render_notebook()

### 涟漪散点图（EffectScatter）

涟漪散点图带有动态涟漪效果，适合突出显示重点数据。

In [90]:
from pyecharts import options as opts
from pyecharts.charts import EffectScatter
from pyecharts.globals import SymbolType

# 涟漪散点图：城市GDP排名
escatter = (
    EffectScatter(init_opts=opts.InitOpts(width="700px", height="400px"))
    .add_xaxis(["北京", "上海", "广州", "深圳", "成都", "杭州"])
    .add_yaxis(
        "GDP排名",
        [1, 2, 3, 4, 5, 6],
        symbol=SymbolType.DIAMOND,
        symbol_size=20
    )
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show=True, position="top"),
        effect_opts=opts.EffectOpts(scale=3, period=2, color="#FF6B6B")
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="城市GDP排名（涟漪效果）"),
        tooltip_opts=opts.TooltipOpts(trigger="item"),
        xaxis_opts=opts.AxisOpts(name="城市"),
        yaxis_opts=opts.AxisOpts(
            name="排名",
            is_inverse=True
        )
    )
)

escatter.render_notebook()

### 水球图（Liquid）

水球图适合展示百分比或进度类的数据。

In [20]:
# 水球图：项目完成率
liquid = (
    Liquid(init_opts=opts.InitOpts(width="400px", height="400px"))
    .add("完成率", [0.78], is_animation=True, color=["#4ECDC4"])
    .set_global_opts(
        title_opts=opts.TitleOpts(title="项目完成率", pos_left="center")
    )
)
liquid.render_notebook()

### 仪表盘（Gauge）

仪表盘适合展示单一指标的当前值与目标范围的关系。

In [92]:
# 仪表盘：用户满意度评分
gauge = (
    Gauge(init_opts=opts.InitOpts(width="500px", height="400px"))
    .add(
        "满意度",
        [("评分", 85)],
        split_number=10
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="用户满意度评分", pos_left="center"),
        tooltip_opts=opts.TooltipOpts(formatter="{a} <br/>{b} : {c}%")
    )
)
gauge.render_notebook()

### 旭日图（Sunburst）

旭日图是一种高级饼图，可以展示具有层级关系的数据。

In [93]:
# 旭日图：企鹅分类
data = [
    {"name": "企鹅", "children": [
        {"name": "Adelie", "value": 150},
        {"name": "Gentoo", "value": 120},
        {"name": "Chinstrap", "value": 68}
    ]},
    {"name": "栖息地", "children": [
        {"name": "Torgersen", "value": 50},
        {"name": "Biscoe", "value": 100},
        {"name": "Dream", "value": 88}
    ]}
]

sunburst = (
    Sunburst(init_opts=opts.InitOpts(width="600px", height="500px"))
    .add("", data)
    .set_global_opts(
        title_opts=opts.TitleOpts(title="企鹅分类旭日图", pos_left="center")
    )
)
sunburst.render_notebook()

## 8.6 主题与样式

PyEcharts 支持多种内置主题，可以一键切换图表的整体风格。

In [95]:
# 不同主题的折线图示例
# 主题列表：LIGHT, DARK, ESSOS, WALDEN, CHALK, INFOGRAPHIC, MACARONS, PURPLE_PASSION 等

themes = [ThemeType.LIGHT, ThemeType.DARK, ThemeType.MACARONS]
for theme in themes:
    line_theme = (
        Line(init_opts=opts.InitOpts(theme=theme, width="600px", height="300px"))
        .add_xaxis(["周一", "周二", "周三", "周四", "周五"])
        .add_yaxis("销量", [120, 150, 90, 180, 200])
        .add_yaxis("利润", [80, 100, 60, 120, 140])
        .set_global_opts(
            title_opts=opts.TitleOpts(title=f"主题演示: {theme}"),
            tooltip_opts=opts.TooltipOpts(trigger="axis")
        )
    )
    display(line_theme.render_notebook())
    print(f"--- {theme} 主题 ---")

--- light 主题 ---


--- dark 主题 ---


--- macarons 主题 ---


### 自定义配色

可以通过 color 参数为图表设置自定义配色方案。

In [99]:
# 自定义配色饼图
custom_colors = ["#7FB3D5", "#76D7C4", "#F7DC6F", "#F8C471", "#C39BD3", "#A3E4D7"]

pie_color = (
    Pie(init_opts=opts.InitOpts(width="600px", height="400px"))
    .add(
        "",
        [["学习", 35], ["工作", 40], ["娱乐", 15], ["休息", 10]],
        radius=["35%", "65%"],
        color=custom_colors,
        label_opts=opts.LabelOpts(formatter="{b}: {d}%")
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="时间分配饼图", pos_left="center"),
        legend_opts=opts.LegendOpts(orient="vertical", pos_left="left", pos_top="20%")
    )
)

pie_color.render_notebook()

### DataZoom 数据区域缩放

DataZoom 组件允许用户通过拖拽或滑块来缩放数据区域，适合大数据量的展示。

In [25]:
# 数据区域缩放
import random

dates = [f"2024-{i//28+1:02d}-{i%28+1:02d}" for i in range(360)]
values = [100 + i * 0.5 + random.randint(-20, 20) for i in range(360)]

line_zoom = (
    Line(init_opts=opts.InitOpts(width="900px", height="400px"))
    .add_xaxis(dates)
    .add_yaxis("访问量", values)
    .set_global_opts(
        title_opts=opts.TitleOpts(title="日访问量趋势（可缩放）"),
        xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
        datazoom_opts=opts.DataZoomOpts(
            range_start=0,
            range_end=30,
            orient="horizontal",
            pos_top="85%"
        ),
        tooltip_opts=opts.TooltipOpts(trigger="axis")
    )
)
line_zoom.render_notebook()

## 8.7 总结

本教程介绍了 PyEcharts 的核心功能和用法，包括基础图表、数据可视化、Grid 布局、组合组件和高级图表。

### 核心组件速查

| 组件 | 用途 |
|------|------|
| Grid | 多图表网格布局，支持精确定位 |
| Page | 多图表分页展示 |
| Tab | 多图表标签页切换 |
| Timeline | 时间线动画展示 |
| Overlap | 图表叠加（折线+柱状混合）|

### Grid 防重叠要点

1. **上下布局**：上方图表设置 `pos_bottom="52%"`，下方图表设置 `pos_top="55%"`
2. **左右布局**：左侧图表设置 `pos_right="52%"`，右侧图表设置 `pos_left="55%"`
3. **四宫格**：每个格子约 42-45%，间距约 10%，确保 `pos_top + pos_bottom < 100%`
4. **使用 grid_gap**：增加格子之间的间距

### 关键函数参考

```python
# 图表初始化
chart = Line(init_opts=opts.InitOpts(width="800px", height="400px", theme=ThemeType.LIGHT))

# 添加数据
chart.add_xaxis([...])
chart.add_yaxis("系列名", [...], symbol="circle", symbol_size=10)

# 全局配置
chart.set_global_opts(
    title_opts=opts.TitleOpts(title="标题", subtitle="副标题"),
    tooltip_opts=opts.TooltipOpts(trigger="item"),
    legend_opts=opts.LegendOpts(pos_left="left"),
    xaxis_opts=opts.AxisOpts(name="X轴", axislabel_opts=opts.LabelOpts(rotate=30)),
    yaxis_opts=opts.AxisOpts(name="Y轴")
)

# Grid 布局
grid = Grid().add(chart, grid_opts=opts.GridOpts(pos_top="10%", pos_bottom="10%", pos_left="10%", pos_right="10%"))

# 渲染
chart.render("output.html")  # 保存HTML
chart.render_notebook()       # Jupyter中显示
```

### 继续学习

- PyEcharts 官方文档：https://pyecharts.org
- ECharts 官方示例：https://echarts.apache.org/examples/
- 更多主题：ThemeType.LIGHT, DARK, CHALK, WALDEN, INFOGRAPHIC 等
- 更多图表：Graph（关系图）、TreeMap（树图）、Parallel（平行坐标）等